# PowerPulse v2 — faster electricity-bill OCR + translation + transliteration

Run the cells top to bottom. Everything you may want to change is in **Cell 1 (Settings)**.

## What is faster in this version
| Step | Before | Now |
|---|---|---|
| Install | All 10 OCR packs + IndicXlit (fairseq, very heavy) + IndicTransToolkit every time | Only the OCR packs you use. Model packages only if you pick a model translation mode. Skipped if already installed |
| OCR | One bill at a time, all 10 Indian languages on every bill | Bills run in parallel, script detection narrows each bill to one language, adaptive image scaling, results cached by image hash (re-runs are instant) |
| Translation | Every OCR line sent to IndicTrans2 one by one (batch size 1) | Bill-label glossary first (instant), bilingual lines need no translation, only the leftovers go to the model, in sorted batches |
| Names / addresses | IndicXlit neural model | **`indic-transliteration`** (rule-based, no model download, milliseconds) with Hindi-style schwa deletion |

## Bug fixes
- Hindi address / previous-balance patterns mixed Telugu and Devanagari letters (`పता`, `పिछला`), so they never matched. Fixed.
- Consumer-number pattern now accepts the OCR spelling `उपभोका`.
- Final cell printed literal `\\n`. Fixed.

## What is new in v3 — all Indian languages, automatic language detection
| Area | v3 |
|---|---|
| Language detection | Automatic per bill: script first, then marker words/letters to tell apart languages that share a script (Hindi / Marathi / Nepali, Bengali / Assamese, Urdu / Sindhi / Kashmiri). Result is stored in `source_language`, `source_script`, `language_detection_method`, `language_detection_confidence` |
| OCR | Packs for Hindi, Marathi, Nepali, Bengali, Assamese, Gujarati, Punjabi, Odia, Tamil, Telugu, Kannada, Malayalam, Urdu. Script detection still picks **one** language per bill, so OCR time does not grow. A bill detected as Marathi/Nepali/Assamese is re-read once with the right pack |
| Output | JSON is English. Original OCR text is kept untouched; every non-English line is also listed in `translated_lines` as original, English translation and transliteration. Names and addresses carry original + transliteration (+ translation for addresses) in `text_fields` |
| Bill-label glossary | Hindi, Telugu, Kannada (as before) plus Marathi, Bengali, Gujarati, Tamil, Malayalam, Punjabi, Odia, Urdu |
| Model | Still never loaded in `"fast"` mode, except for lines the rule-based transliterator cannot handle (Urdu, Sindhi, Kashmiri, Santali, Manipuri) |

OCR output is never guaranteed correct. Check amounts, dates and names against the original bill images.

## Cell 1 — Settings

In [ ]:
# ============================ SETTINGS ============================
# "fast"   = glossary + transliteration only. No AI model is loaded. Fastest.
# "hybrid" = glossary first, then IndicTrans2 only for lines still left in an Indian script.
# "model"  = IndicTrans2 for every Indian-script line (slowest, closest to the old notebook).
TRANSLATION_MODE = "fast"

# Fallback OCR languages, used ONLY when the script of a bill cannot be detected (English is always on).
# Options: hin mar nep ben asm guj pan ori tam tel kan mal urd
OCR_LANGUAGES = ["hin", "tel", "kan"]

INSTALL_ALL_INDIAN_OCR = True    # one-time install of OCR packs for every supported Indian language.
                                 # OCR itself still uses only the language detected for each bill (no slowdown).
                                 # Set False to install just OCR_LANGUAGES (faster first install).
AUTO_MODEL_FOR_UNCOVERED = True  # "fast" mode only: lines in scripts the rule-based transliterator cannot do
                                 # (Urdu/Sindhi/Kashmiri, Santali, Manipuri) are translated by IndicTrans2.
                                 # Bills in other scripts never load a model in "fast" mode.

AUTO_NARROW_LANGUAGE = True      # detect the script per bill and OCR with just that language
SPLIT_COMPOSITE = True           # split images whose name contains COMPOSITE_KEYWORD into 3 bills
COMPOSITE_KEYWORD = "composite"
MAX_WORKERS = None               # None = use all CPU cores
USE_CACHE = True                 # reuse OCR results for identical images
CPU_QUANTIZE = True              # int8 speed-up for model modes when no GPU is available
LOCAL_FOLDER = None              # e.g. "/content/bills" to skip the upload dialog
CACHE_DIR = "/content/powerpulse_cache"
OUTPUT_DIR = "/content/electricity_bill_output"
# ==================================================================

## Cell 2 — Install only what is needed (skipped when already installed)

In [ ]:
import importlib.util, shutil, subprocess, sys, time
_t0 = time.time()

TESS_PACKS = {"hin": "tesseract-ocr-hin", "tel": "tesseract-ocr-tel", "kan": "tesseract-ocr-kan",
              "mal": "tesseract-ocr-mal", "urd": "tesseract-ocr-urd", "ben": "tesseract-ocr-ben",
              "guj": "tesseract-ocr-guj", "pan": "tesseract-ocr-pan", "ori": "tesseract-ocr-ori",
              "tam": "tesseract-ocr-tam", "mar": "tesseract-ocr-mar", "nep": "tesseract-ocr-nep",
              "asm": "tesseract-ocr-asm"}

def _installed_tess_langs():
    if not shutil.which("tesseract"):
        return set()
    out = subprocess.run(["tesseract", "--list-langs"], capture_output=True, text=True)
    return {l.strip() for l in (out.stdout + out.stderr).splitlines()[1:] if l.strip()}

def _have(module):
    return importlib.util.find_spec(module) is not None

INSTALL_LANGS = list(TESS_PACKS) if INSTALL_ALL_INDIAN_OCR else [c for c in OCR_LANGUAGES if c in TESS_PACKS]
have = _installed_tess_langs()
wanted = {"eng", "osd"} | set(INSTALL_LANGS) | set(OCR_LANGUAGES)
if not wanted <= have:
    _apt = ["apt-get", "-qq", "install", "-y", "--no-install-recommends"]
    pkgs = ["tesseract-ocr", "tesseract-ocr-osd"] + [TESS_PACKS[c] for c in INSTALL_LANGS if c not in have]
    if subprocess.run(_apt + pkgs, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
        if subprocess.run(_apt + pkgs, capture_output=True).returncode != 0:
            # a pack missing from this Ubuntu release must not block the others: install them one by one
            subprocess.run(_apt + ["tesseract-ocr", "tesseract-ocr-osd"], check=True, capture_output=True)
            for p in pkgs[2:]:
                if subprocess.run(_apt + [p], capture_output=True).returncode != 0:
                    print("OCR pack not available, skipped:", p)

pip_needed = [p for p, m in [("pytesseract", "pytesseract"), ("Pillow", "PIL"), ("numpy", "numpy"),
                             ("indic-transliteration", "indic_transliteration")] if not _have(m)]
if TRANSLATION_MODE in ("hybrid", "model"):
    pip_needed += [p for p, m in [("transformers", "transformers"), ("sentencepiece", "sentencepiece"),
                                  ("sacremoses", "sacremoses")] if not _have(m)]
if pip_needed:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install"] + pip_needed, check=True)
if TRANSLATION_MODE in ("hybrid", "model") and not _have("IndicTransToolkit"):
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "git+https://github.com/VarunGumma/IndicTransToolkit.git"], check=True)

print(f"Install step finished in {time.time() - _t0:.1f}s | OCR languages ready: {sorted(_installed_tess_langs())}")

## Cell 3 — Upload the bill images

In [ ]:
from pathlib import Path

IMAGE_EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
uploaded_images = {}

if LOCAL_FOLDER:
    for p in sorted(Path(LOCAL_FOLDER).iterdir()):
        if p.suffix.lower() in IMAGE_EXT:
            uploaded_images[p.name] = p.read_bytes()
else:
    from google.colab import files
    for name, content in files.upload().items():
        if Path(name).suffix.lower() in IMAGE_EXT:
            uploaded_images[Path(name).name] = content
        else:
            print("Skipped (not an image):", name)

if not uploaded_images:
    raise ValueError("No images found. Upload bill images (png/jpg/jpeg/webp).")
print("Images in this session:", len(uploaded_images))
for n in uploaded_images:
    print("-", n)

## Cell 4 — Helpers: Indic transliteration + bill-label glossary

`indic-transliteration` converts Devanagari, Bengali, Gurmukhi, Gujarati, Odia, Tamil, Telugu, Kannada and Malayalam into Roman letters. For Hindi-family scripts a schwa-deletion step turns `ramesha kumāra` into `Ramesh Kumar`. Urdu, Sindhi and Kashmiri (Arabic script), Santali (Ol Chiki) and Manipuri (Meitei Mayek) are not supported by that library; in `"fast"` mode those lines go to IndicTrans2 automatically (`AUTO_MODEL_FOR_UNCOVERED`). If the model cannot be loaded, the original text is kept and flagged.

In [ ]:
import re, time, unicodedata

try:
    from indic_transliteration.sanscript import transliterate as _sanscript
except Exception as _e:
    _sanscript = None
    print("WARNING: indic-transliteration is not available:", _e)

# (start, end, sanscript scheme, apply Hindi-style schwa deletion, language name)
SCRIPT_BLOCKS = [
    (0x0900, 0x097F, "devanagari", True,  "Hindi"),
    (0x0980, 0x09FF, "bengali",    True,  "Bengali"),
    (0x0A00, 0x0A7F, "gurmukhi",   True,  "Punjabi"),
    (0x0A80, 0x0AFF, "gujarati",   True,  "Gujarati"),
    (0x0B00, 0x0B7F, "oriya",      False, "Odia"),
    (0x0B80, 0x0BFF, "tamil",      False, "Tamil"),
    (0x0C00, 0x0C7F, "telugu",     False, "Telugu"),
    (0x0C80, 0x0CFF, "kannada",    False, "Kannada"),
    (0x0D00, 0x0D7F, "malayalam",  False, "Malayalam"),
    (0x1C50, 0x1C7F, None,         False, "Santali"),     # Ol Chiki: detection + model translation only
    (0xABC0, 0xABFF, None,         False, "Manipuri"),    # Meitei Mayek: detection + model translation only
]
INDIC_CHARS = r"\u0900-\u0d7f\u1c50-\u1c7f\uabc0-\uabff"
INDIC_RE = re.compile(f"[{INDIC_CHARS}]")
ARABIC_RE = re.compile(r"[\u0600-\u06ff]")
TOKEN_RE = re.compile(f"[{INDIC_CHARS}]+|[\u0600-\u06ff]+")

_C = r"(?:kh|gh|ch|jh|ṭh|ḍh|th|dh|ph|bh|[kgcjṭḍṇtdnpbmyrlvśṣsḷhṅñ])"
_V = r"(?:ai|au|[aāiīuūeoṛṝ])"
_NAME_FIXES = {"simh": "singh", "sinh": "singh", "indaur": "indore", "lakhnau": "lucknow",
               "lakhanau": "lucknow", "dilli": "delhi", "dili": "delhi"}

def _roman_cleanup(iast, schwa):
    """IAST (with diacritics) -> plain readable Roman letters."""
    words = []
    for w in unicodedata.normalize("NFC", iast).lower().split(" "):
        if schwa and len(re.findall(_V, w)) >= 2:
            if not re.search(rf"{_C}[rylv]a$", w):                      # keep chandra, indra, mitra
                w = re.sub(rf"({_C})a$", r"\1", w)                     # ramesha -> ramesh
            w = re.sub(rf"({_V}{_C})a(?={_C}{_V})", r"\1", w)          # lakhanau -> lakhnau
        w = w.replace("ś", "sh").replace("ṣ", "sh").replace("ṛ", "ri").replace("ṝ", "ri")
        w = w.replace("ch", "\x00").replace("c", "ch").replace("\x00", "chh")
        w = re.sub(r"[ṃṁ](?=[pbm])|[ṃṁ]$", "m", w)
        w = re.sub(r"[ṃṁ]", "n", w)
        w = "".join(ch for ch in unicodedata.normalize("NFKD", w) if unicodedata.category(ch) != "Mn")
        words.append(_NAME_FIXES.get(w, w))
    return " ".join(words)

_xlit_engine = None
def _indicxlit_word(token, lang_code="ur"):
    """Optional fallback (Urdu). Only works if you pip-installed ai4bharat-transliteration yourself."""
    global _xlit_engine
    try:
        if _xlit_engine is None:
            from ai4bharat.transliteration import XlitEngine
            _xlit_engine = XlitEngine(src_script_type="indic", beam_width=4, rescore=False)
        res = _xlit_engine.translit_word(token, lang_code=lang_code, topk=1)
        if isinstance(res, dict):
            res = res.get(lang_code) or []
        return res[0] if res else token
    except Exception:
        return token

def _token_to_roman(match):
    token = match.group(0)
    if ARABIC_RE.match(token):
        return _indicxlit_word(token)
    cp = ord(token[0])
    for lo, hi, scheme, schwa, _name in SCRIPT_BLOCKS:
        if lo <= cp <= hi:
            if _sanscript is None or scheme is None:
                return token
            try:
                iast = _sanscript(token, scheme, "iast")
                if INDIC_RE.search(iast):          # library could not convert it: keep the original word
                    return token
                return _roman_cleanup(iast, schwa)
            except Exception:
                return token
    return token

def transliterate_text(value):
    """Convert every Indian-script word in `value` to Roman letters; keep spaces, digits, punctuation."""
    if value is None:
        return None
    value = ascii_digits(str(value).strip())
    if not value or not (INDIC_RE.search(value) or ARABIC_RE.search(value)):
        return value or None
    out = TOKEN_RE.sub(_token_to_roman, value)
    out = re.sub(r"\s+", " ", out).strip()
    return " ".join(w[:1].upper() + w[1:] if w.islower() else w for w in out.split(" "))

# ---- Bill-label glossary: instant translation of the words that appear on every bill ----------
GLOSSARY = {
    # Hindi
    "बिजली बिल": "Electricity bill", "विद्युत देयक": "Electricity bill", "उपभोक्ता संख्या": "Consumer number",
    "उपभोका संख्या": "Consumer number", "उपभोक्ता क्रमांक": "Consumer number", "उपभोका क्रमांक": "Consumer number",
    "ग्राहक आईडी": "Customer ID", "बिल माह": "Bill month", "बिल दिनांक": "Bill date", "देय तिथि": "Due date",
    "नाम": "Name", "पता": "Address", "वितरण खंड": "Distribution division", "विभाग": "Department",
    "वृत्त": "Circle", "कनेक्शन प्रकार": "Connection type", "कनेकशन प्रकार": "Connection type",
    "कनेक्शन श्रेणी": "Connection category", "कनेकशन श्रेणी": "Connection category", "टैरिफ श्रेणी": "Tariff category",
    "घरेलू": "Domestic", "एकल फेज": "single phase", "लोड": "Load", "सप्लाई वोल्टेज": "Supply voltage",
    "सुरक्षा जमा": "Security deposit", "मीटर संख्या": "Meter number", "मीटर रीडिंग विवरण": "Meter reading details",
    "मीटर विवरण": "Meter details", "वर्तमान रीडिंग": "Current reading", "पूर्व रीडिंग": "Previous reading",
    "उपयोग (युनिट)": "Usage (units)", "उपयोग (यूनिट)": "Usage (units)", "कुल खपत (युनिट)": "Total consumption (units)",
    "कुल यूनिट": "Total units", "कुल युनिट": "Total units", "खपत (युनिट)": "Consumption (units)", "खपत": "Consumption",
    "गुणक": "Multiplier", "दिनों की संख्या": "Number of days", "दिनों की": "Days",
    "ऊर्जा शुल्क": "Energy charges", "ऊर्जा प्रभार": "Energy charges", "स्थायी शुल्क": "Fixed charges",
    "स्थयी शुल्क": "Fixed charges", "निश्चित शुल्क": "Fixed charges", "निश्वित शुल्क": "Fixed charges",
    "विद्युत कर": "Electricity tax", "विद्युत शुल्क": "Electricity duty",
    "ईंधन समायोजन शुल्क": "Fuel adjustment charge", "ईधन समायोजन शुल्क": "Fuel adjustment charge",
    "अन्य समायोजन": "Other adjustments", "कर/अन्य शुल्क": "Tax/other charges", "मीटर रेंट": "Meter rent",
    "कुल देय राशि (वर्तमान माह)": "Total amount due (current month)", "कुल वर्तमान देय": "Total current due",
    "कुल देय राशि": "Total amount due", "पूर्व बकाया राशि": "Previous arrears", "पूर्व बकाया": "Previous arrears",
    "विलंब शुल्क": "Late fee", "अंतिम भुगतान विवरण": "Last payment details",
    "अंतिम भुगतान दिनांक": "Last payment date", "अंतिम भुगतान राशि": "Last payment amount",
    "देय तिथि तक जमा करें": "Pay by the due date", "ग्राहक सेवा केंद्र": "Customer service centre",
    "बिल विवरण": "Bill details", "विवरण": "Description", "राशि": "Amount", "हेल्पलाइन": "helpline",
    "जनवरी": "January", "फरवरी": "February", "मार्च": "March", "अप्रैल": "April", "मई": "May", "जून": "June",
    "जुलाई": "July", "अगस्त": "August", "सितम्बर": "September", "सितंबर": "September", "सितमबर": "September",
    "अक्टूबर": "October", "नवंबर": "November", "नवम्बर": "November", "दिसंबर": "December", "दिसम्बर": "December",
    # Telugu
    "వినియోగపు / కనీస చార్జీలు": "Consumption / minimum charges", "వినియోగపు చార్జీలు": "Consumption charges",
    "స్థిర చార్జీలు": "Fixed charges", "కస్టమర్ చార్జీలు": "Customer charges",
    "కెపాసిటర్ సర్చార్జీ": "Capacitor surcharge", "కెపాసిటర్ సర్ఛార్జీ": "Capacitor surcharge",
    "విద్యుత్ సుంకముపై వడ్డీ": "Interest on electricity duty", "విద్యుత్ సుంకము": "Electricity duty",
    "ఆలస్యమునకు సర్చార్జీలు": "Late payment surcharges", "ఇతర చార్జీలు": "Other charges",
    "సర్దుబాటు మొత్తము": "Adjustment total", "సబ్సిడీ మొత్తము": "Subsidy amount",
    "సవరణ": "Revision", "నెల మొత్తము": "Month total", "మార్చి 31 వరకు బకాయిలు": "Arrears up to March 31",
    "ఏప్రిల్ 1 నుంచి బకాయిలు": "Arrears from April 1", "చెల్లించవలసిన మొత్తము": "Amount payable",
    "గడువు తేది": "Due date", "బిల్ యూనిట్లు": "Bill units", "సరాసరి": "Average",
    "సోలార్ యూనిట్స్": "Solar units", "నెట్ యూనిట్స్": "Net units", "పాత మీటరు": "Old meter",
    "బిల్ ఎం.డి.": "Bill MD", "పి.ఎఫ్": "PF", "విద్యుత్ చార్జీలు": "Electricity charges",
    "వినియోగదారుని సంఖ్య": "Consumer number", "పేరు": "Name", "చిరునామా": "Address",
    "మీటర్ నెం": "Meter no", "బిల్లు తేదీ": "Bill date", "చివరి చెల్లింపు తేదీ": "Last payment date",
    "గత బకాయిలు": "Previous arrears",
    # Kannada
    "ಬಿಲ್ ಸಂಖ್ಯೆ": "Bill no", "ಇಂದಿನ ಮಾಪನ": "Present reading", "ಹಿಂದಿನ ಮಾಪನ": "Previous reading",
    "ನಿಗದಿತ ಶುಲ್ಕ": "Fixed charges", "ವಿದ್ಯುತ್ ಶುಲ್ಕ": "Energy charges", "ತೆರಿಗೆ": "Tax", "ಬಾಕಿ": "Arrears",
    "ಪಾವತಿಯ ಮೊತ್ತ": "Net payable", "ಪಾವತಿಗೆ ಕಡೆಯ ದಿನಾಂಕ": "Due date",
    "ಪ್ರಸ್ತುತ ಬೇಡಿಕೆ": "Current demand payable", "ಬಡ್ಡಿ": "Interest", "ಇತರೆ": "Others", "ಬಳಕೆ": "Consumption",
}

# ---- v3: bill labels for more Indian languages. Values reuse the same English labels the field patterns in Cell 7 expect.
# Original (Hindi / Telugu / Kannada) entries above always win; these only add new keys.
GLOSSARY_EXTRA = {
    # Marathi (Devanagari)
    "वीज बिल": "Electricity bill", "वीजबिल": "Electricity bill", "ग्राहक क्रमांक": "Consumer number",
    "ग्राहक नंबर": "Consumer number", "बिल तारीख": "Bill date", "बिलाची तारीख": "Bill date",
    "देय दिनांक": "Due date", "देय तारीख": "Due date", "भरणा करण्याची अंतिम तारीख": "Due date",
    "नाव": "Name", "पत्ता": "Address", "मीटर क्रमांक": "Meter number", "मीटर नंबर": "Meter number",
    "चालू वाचन": "Current reading", "मागील वाचन": "Previous reading", "चालू रीडिंग": "Current reading",
    "मागील रीडिंग": "Previous reading", "वीज वापर (युनिट)": "Consumption (units)", "एकूण वापर": "Total consumption",
    "वीज वापर": "Consumption", "वापर": "Consumption", "स्थिर आकार": "Fixed charges", "वीज आकार": "Energy charges",
    "वहन आकार": "Wheeling charges", "इंधन समायोजन आकार": "Fuel adjustment charge", "मागील थकबाकी": "Previous arrears",
    "थकबाकी": "Arrears", "व्याज": "Interest", "विलंब आकार": "Late fee", "एकूण देय रक्कम": "Total amount due",
    "चालू बिल": "Current bill", "एकूण बिल": "Total bill", "एकूण": "Total", "रक्कम": "Amount",
    "शेवटचा भरणा": "Last payment", "सुरक्षा ठेव": "Security deposit", "मंजूर भार": "Sanctioned load",
    # Bengali
    "বিদ্যুৎ বিল": "Electricity bill", "গ্রাহক নম্বর": "Consumer number", "গ্রাহক আইডি": "Customer ID",
    "বিলের তারিখ": "Bill date", "বিল তারিখ": "Bill date", "পরিশোধের শেষ তারিখ": "Due date", "শেষ তারিখ": "Due date",
    "নাম": "Name", "ঠিকানা": "Address", "মিটার নম্বর": "Meter number", "বর্তমান রিডিং": "Current reading",
    "পূর্ববর্তী রিডিং": "Previous reading", "ব্যবহার (ইউনিট)": "Consumption (units)", "মোট ইউনিট": "Total units",
    "স্থির চার্জ": "Fixed charges", "শক্তি চার্জ": "Energy charges", "বিদ্যুৎ শুল্ক": "Electricity duty",
    "পূর্ববর্তী বকেয়া": "Previous arrears", "বকেয়া": "Arrears", "বিলম্ব মাশুল": "Late fee", "সুদ": "Interest",
    "মোট প্রদেয় অর্থ": "Total amount due", "মোট": "Total", "পরিমাণ": "Amount",
    # Gujarati
    "વીજ બિલ": "Electricity bill", "ગ્રાહક નંબર": "Consumer number", "બિલ તારીખ": "Bill date",
    "બિલની તારીખ": "Bill date", "નિયત તારીખ": "Due date", "ભરવાની છેલ્લી તારીખ": "Due date",
    "નામ": "Name", "સરનામું": "Address", "મીટર નંબર": "Meter number", "વર્તમાન રીડિંગ": "Current reading",
    "પાછલું રીડિંગ": "Previous reading", "વપરાશ (યુનિટ)": "Consumption (units)", "કુલ યુનિટ": "Total units",
    "વપરાશ": "Consumption", "ફિક્સ ચાર્જ": "Fixed charges", "એનર્જી ચાર્જ": "Energy charges",
    "વીજળી શુલ્ક": "Electricity duty", "અગાઉનું બાકી": "Previous arrears", "બાકી": "Arrears", "વ્યાજ": "Interest",
    "કુલ ચૂકવવાપાત્ર રકમ": "Total amount due", "કુલ": "Total", "રકમ": "Amount",
    # Tamil
    "மின் கட்டண ரசீது": "Electricity bill", "மின் கட்டணம்": "Electricity charges",
    "மின் இணைப்பு எண்": "Service connection number", "நுகர்வோர் எண்": "Consumer number",
    "பில் தேதி": "Bill date", "செலுத்த வேண்டிய கடைசி நாள்": "Due date", "கடைசி தேதி": "Due date",
    "பெயர்": "Name", "முகவரி": "Address", "மீட்டர் எண்": "Meter number", "தற்போதைய அளவீடு": "Current reading",
    "முந்தைய அளவீடு": "Previous reading", "நுகர்வு (யூனிட்)": "Consumption (units)", "மொத்த யூனிட்": "Total units",
    "நுகர்வு": "Consumption", "நிலைக் கட்டணம்": "Fixed charges", "நிலை கட்டணம்": "Fixed charges",
    "மின்சார வரி": "Electricity tax", "நிலுவைத் தொகை": "Arrears", "நிலுவை": "Arrears", "வட்டி": "Interest",
    "செலுத்த வேண்டிய மொத்த தொகை": "Total amount due", "மொத்தம்": "Total", "தொகை": "Amount",
    # Malayalam
    "വൈദ്യുതി ബിൽ": "Electricity bill", "കൺസ്യൂമർ നമ്പർ": "Consumer number", "ഉപഭോക്തൃ നമ്പർ": "Consumer number",
    "ബിൽ തീയതി": "Bill date", "അവസാന തീയതി": "Due date", "പേര്": "Name", "വിലാസം": "Address",
    "മീറ്റർ നമ്പർ": "Meter number", "നിലവിലെ റീഡിംഗ്": "Current reading", "മുൻ റീഡിംഗ്": "Previous reading",
    "ഉപഭോഗം (യൂണിറ്റ്)": "Consumption (units)", "ഉപഭോഗം": "Consumption", "ഫിക്സഡ് ചാർജ്": "Fixed charges",
    "എനർജി ചാർജ്": "Energy charges", "വൈദ്യുതി ഡ്യൂട്ടി": "Electricity duty", "കുടിശ്ശിക": "Arrears",
    "പലിശ": "Interest", "ആകെ അടയ്ക്കേണ്ട തുക": "Total amount due", "ആകെ": "Total", "തുക": "Amount",
    # Punjabi (Gurmukhi)
    "ਬਿਜਲੀ ਬਿੱਲ": "Electricity bill", "ਖਪਤਕਾਰ ਨੰਬਰ": "Consumer number", "ਬਿੱਲ ਮਿਤੀ": "Bill date",
    "ਅੰਤਿਮ ਮਿਤੀ": "Due date", "ਨਾਮ": "Name", "ਪਤਾ": "Address", "ਮੀਟਰ ਨੰਬਰ": "Meter number",
    "ਮੌਜੂਦਾ ਰੀਡਿੰਗ": "Current reading", "ਪਿਛਲੀ ਰੀਡਿੰਗ": "Previous reading", "ਖਪਤ (ਯੂਨਿਟ)": "Consumption (units)",
    "ਖਪਤ": "Consumption", "ਫਿਕਸਡ ਚਾਰਜ": "Fixed charges", "ਬਿਜਲੀ ਡਿਊਟੀ": "Electricity duty", "ਬਕਾਇਆ": "Arrears",
    "ਵਿਆਜ": "Interest", "ਕੁੱਲ ਰਕਮ": "Total amount due", "ਕੁੱਲ": "Total", "ਰਕਮ": "Amount",
    # Odia
    "ବିଦ୍ୟୁତ ବିଲ୍": "Electricity bill", "ଗ୍ରାହକ ନମ୍ବର": "Consumer number", "ବିଲ୍ ତାରିଖ": "Bill date",
    "ଶେଷ ତାରିଖ": "Due date", "ନାମ": "Name", "ଠିକଣା": "Address", "ମିଟର ନମ୍ବର": "Meter number",
    "ବର୍ତ୍ତମାନ ରିଡିଂ": "Current reading", "ପୂର୍ବ ରିଡିଂ": "Previous reading", "ବ୍ୟବହାର": "Consumption",
    "ସ୍ଥିର ଶୁଳ୍କ": "Fixed charges", "ବିଦ୍ୟୁତ ଶୁଳ୍କ": "Electricity duty", "ବକେୟା": "Arrears", "ସୁଧ": "Interest",
    "ମୋଟ ଦେୟ": "Total amount due", "ମୋଟ": "Total", "ରାଶି": "Amount",
    # Urdu (Arabic script)
    "بجلی کا بل": "Electricity bill", "صارف نمبر": "Consumer number", "بل کی تاریخ": "Bill date",
    "آخری تاریخ": "Due date", "نام": "Name", "پتہ": "Address", "میٹر نمبر": "Meter number",
    "موجودہ ریڈنگ": "Current reading", "پچھلی ریڈنگ": "Previous reading", "کھپت": "Consumption",
    "فکسڈ چارجز": "Fixed charges", "بقایا": "Arrears", "سود": "Interest",
    "کل واجب الادا رقم": "Total amount due", "کل": "Total", "رقم": "Amount",
}

# Month names (new languages + Telugu/Kannada). Used to find "bill month" and to write it in English.
# One/two-letter names (e.g. May) are only used for the bill-month lookup, never replaced inside free text.
MONTH_WORDS = {
    # Marathi
    "जानेवारी": "January", "फेब्रुवारी": "February", "एप्रिल": "April", "मे": "May", "जुलै": "July",
    "ऑगस्ट": "August", "सप्टेंबर": "September", "ऑक्टोबर": "October", "नोव्हेंबर": "November",
    # Bengali
    "জানুয়ারি": "January", "ফেব্রুয়ারি": "February", "মার্চ": "March", "এপ্রিল": "April", "মে": "May",
    "জুন": "June", "জুলাই": "July", "আগস্ট": "August", "সেপ্টেম্বর": "September", "অক্টোবর": "October",
    "নভেম্বর": "November", "ডিসেম্বর": "December",
    # Gujarati
    "જાન્યુઆરી": "January", "ફેબ્રુઆરી": "February", "માર્ચ": "March", "એપ્રિલ": "April", "મે": "May",
    "જૂન": "June", "જુલાઈ": "July", "ઓગસ્ટ": "August", "સપ્ટેમ્બર": "September", "ઓક્ટોબર": "October",
    "નવેમ્બર": "November", "ડિસેમ્બર": "December",
    # Tamil
    "ஜனவரி": "January", "பிப்ரவரி": "February", "மார்ச்": "March", "ஏப்ரல்": "April", "மே": "May",
    "ஜூன்": "June", "ஜூலை": "July", "ஆகஸ்ட்": "August", "செப்டம்பர்": "September", "அக்டோபர்": "October",
    "நவம்பர்": "November", "டிசம்பர்": "December",
    # Malayalam
    "ജനുവരി": "January", "ഫെബ്രുവരി": "February", "മാർച്ച്": "March", "ഏപ്രിൽ": "April", "മേയ്": "May",
    "ജൂൺ": "June", "ജൂലൈ": "July", "ഓഗസ്റ്റ്": "August", "സെപ്റ്റംബർ": "September", "ഒക്ടോബർ": "October",
    "നവംബർ": "November", "ഡിസംബർ": "December",
    # Punjabi
    "ਜਨਵਰੀ": "January", "ਫ਼ਰਵਰੀ": "February", "ਫਰਵਰੀ": "February", "ਮਾਰਚ": "March", "ਅਪ੍ਰੈਲ": "April",
    "ਮਈ": "May", "ਜੂਨ": "June", "ਜੁਲਾਈ": "July", "ਅਗਸਤ": "August", "ਸਤੰਬਰ": "September",
    "ਅਕਤੂਬਰ": "October", "ਨਵੰਬਰ": "November", "ਦਸੰਬਰ": "December",
    # Odia
    "ଜାନୁଆରୀ": "January", "ଫେବୃଆରୀ": "February", "ମାର୍ଚ୍ଚ": "March", "ଏପ୍ରିଲ": "April", "ମଇ": "May",
    "ଜୁନ": "June", "ଜୁଲାଇ": "July", "ଅଗଷ୍ଟ": "August", "ସେପ୍ଟେମ୍ବର": "September", "ଅକ୍ଟୋବର": "October",
    "ନଭେମ୍ବର": "November", "ଡିସେମ୍ବର": "December",
    # Urdu
    "جنوری": "January", "فروری": "February", "مارچ": "March", "اپریل": "April", "مئی": "May",
    "جون": "June", "جولائی": "July", "اگست": "August", "ستمبر": "September", "اکتوبر": "October",
    "نومبر": "November", "دسمبر": "December",
    # Telugu
    "జనవరి": "January", "ఫిబ్రవరి": "February", "మార్చి": "March", "ఏప్రిల్": "April", "మే": "May",
    "జూన్": "June", "జూలై": "July", "ఆగస్టు": "August", "సెప్టెంబర్": "September", "అక్టోబర్": "October",
    "నవంబర్": "November", "డిసెంబర్": "December",
    # Kannada
    "ಜನವರಿ": "January", "ಫೆಬ್ರವರಿ": "February", "ಮಾರ್ಚ್": "March", "ಏಪ್ರಿಲ್": "April", "ಮೇ": "May",
    "ಜೂನ್": "June", "ಜುಲೈ": "July", "ಆಗಸ್ಟ್": "August", "ಸೆಪ್ಟೆಂಬರ್": "September", "ಅಕ್ಟೋಬರ್": "October",
    "ನವೆಂಬರ್": "November", "ಡಿಸೆಂಬರ್": "December",
}

def _norm(s):
    return unicodedata.normalize("NFC", s).replace("\u200c", "").replace("\u200d", "")

for _k, _v in GLOSSARY_EXTRA.items():          # new languages only add keys; the original entries always win
    GLOSSARY.setdefault(_k, _v)
_GLOSS = {_norm(k): v for k, v in GLOSSARY.items()}
MONTH_WORDS_N = {_norm(k): v for k, v in MONTH_WORDS.items()}
for _k, _v in MONTH_WORDS_N.items():
    if len(_k) > 2:                              # 1-2 letter month names (May) are ambiguous inside free text
        _GLOSS.setdefault(_k, _v)
_B = "\u0900-\u0963\u0966-\u0d7f\u1c50-\u1c7f\uabc0-\uabff\u0600-\u06ff"   # letters/marks, but not the danda (। ॥)

def _script_key(ch):
    """Script block name of one character (Hindi = Devanagari, Urdu = Arabic range) or None."""
    cp = ord(ch)
    for lo, hi, _s, _w, name in SCRIPT_BLOCKS:
        if lo <= cp <= hi:
            return name
    return "Urdu" if 0x0600 <= cp <= 0x06FF else None

# One regex per script: a line is only matched against the glossary of the scripts it actually contains.
_by_script = {}
for _k, _v in _GLOSS.items():
    _first = next((c for c in _k if _script_key(c)), None)
    if _first:
        _by_script.setdefault(_script_key(_first), {})[_k] = _v
_GLOSS_RES = {name: re.compile(f"(?<![{_B}])(?:" + "|".join(re.escape(k) for k in sorted(d, key=len, reverse=True)) + f")(?![{_B}])")
              for name, d in _by_script.items()}

def apply_glossary(line):
    line = _norm(line)
    for name in {_script_key(tok[0]) for tok in TOKEN_RE.findall(line)}:
        rx = _GLOSS_RES.get(name)
        if rx:
            line = rx.sub(lambda m: _GLOSS[m.group(0)], line)
    return line

_DIGIT_MAP = {}
for _base in (0x0966, 0x09E6, 0x0A66, 0x0AE6, 0x0B66, 0x0BE6, 0x0C66, 0x0CE6, 0x0D66, 0x0660, 0x06F0, 0x1C50, 0xABF0):
    for _i in range(10):
        _DIGIT_MAP[_base + _i] = str(_i)
def ascii_digits(s):
    return s.translate(_DIGIT_MAP)

# ---- v3: automatic language detection --------------------------------------------------------------
# Step 1: which script dominates the OCR text (cheap, exact).
# Step 2: for scripts shared by several languages, look for marker words/letters that tell them apart.
LANG_CODES = {"Hindi": "hin_Deva", "Marathi": "mar_Deva", "Nepali": "npi_Deva", "Bengali": "ben_Beng",
              "Assamese": "asm_Beng", "Punjabi": "pan_Guru", "Gujarati": "guj_Gujr", "Odia": "ory_Orya",
              "Tamil": "tam_Taml", "Telugu": "tel_Telu", "Kannada": "kan_Knda", "Malayalam": "mal_Mlym",
              "Urdu": "urd_Arab", "Sindhi": "snd_Arab", "Kashmiri": "kas_Arab", "Santali": "sat_Olck",
              "Manipuri": "mni_Mtei", "English": "eng_Latn"}
SCRIPT_NAMES = {"Hindi": "Devanagari", "Bengali": "Bengali", "Punjabi": "Gurmukhi", "Gujarati": "Gujarati",
                "Odia": "Odia", "Tamil": "Tamil", "Telugu": "Telugu", "Kannada": "Kannada",
                "Malayalam": "Malayalam", "Urdu": "Arabic", "Santali": "Ol Chiki", "Manipuri": "Meitei Mayek"}

def _word_re(words):
    return re.compile(f"(?<![{_B}])(?:" + "|".join(re.escape(_norm(w)) for w in sorted(words, key=len, reverse=True)) + f")(?![{_B}])")

_MARKERS = {   # Devanagari-script languages that we can tell apart from words alone
    "Marathi": _word_re(["आहे", "आणि", "वीज", "वीजबिल", "महावितरण", "रक्कम", "एकूण", "थकबाकी", "नाव", "पत्ता",
                         "मागील", "वाचन", "भरणा", "वापर", "ग्राहक क्रमांक"]),
    "Nepali": _word_re(["बिजुली", "महसुल", "मिति", "रकम", "ठेगाना", "गर्नुहोस्", "ग्राहक नं"]),
    "Hindi": _word_re(["बिजली", "उपभोक्ता", "उपभोका", "राशि", "तिथि", "देय", "है", "का", "की", "के", "पता",
                       "नाम", "कुल", "पिछला", "पूर्व", "माह"]),
}
_ASSAMESE_LETTERS = set("\u09f0\u09f1")              # ৰ ৱ exist only in Assamese
_SINDHI_LETTERS = set("ڳڄڏڌٺٻڀڦڙڪڻڱڃڇ")              # Sindhi-only Arabic-script letters
_KASHMIRI_LETTERS = set("ۄۆٲۍێ")                      # Kashmiri-only letters

def _script_counts(text):
    counts = {}
    for ch in text:
        if ord(ch) >= 0x0600:
            name = _script_key(ch)
            if name:
                counts[name] = counts.get(name, 0) + 1
    return counts

def detect_language_details(text):
    """Dominant language of `text` -> {language, code (IndicTrans2), script, script_block, method, confidence}."""
    text = _norm(text or "")
    counts = _script_counts(text)
    if not counts:
        return {"language": "English", "code": "eng_Latn", "script": "Latin", "script_block": None,
                "method": "no Indian-script text found", "confidence": "high"}
    block = max(counts, key=counts.get)
    lang, method, conf = block, "script", "high"
    if block == "Hindi":                                   # Devanagari: Hindi / Marathi / Nepali
        hits = {k: len(rx.findall(text)) for k, rx in _MARKERS.items()}
        if hits["Marathi"] >= 2 and hits["Marathi"] > hits["Hindi"] and hits["Marathi"] >= hits["Nepali"]:
            lang, method, conf = "Marathi", "script + marker words", "medium"
        elif hits["Nepali"] >= 2 and hits["Nepali"] > hits["Hindi"]:
            lang, method, conf = "Nepali", "script + marker words", "medium"
        elif hits["Hindi"] >= 2:
            method, conf = "script + marker words", "high"
        elif hits["Hindi"] == 1:
            method, conf = "script + marker words", "medium"
        else:
            method, conf = "script only (Devanagari: Hindi assumed; Konkani/Maithili/Dogri/Bodo/Sanskrit look the same)", "low"
    elif block == "Bengali":                               # Bengali script: Bengali / Assamese
        if sum(1 for ch in text if ch in _ASSAMESE_LETTERS) >= 2:
            lang, method, conf = "Assamese", "script + Assamese-only letters", "medium"
    elif block == "Urdu":                                  # Arabic script: Urdu / Sindhi / Kashmiri
        if sum(1 for ch in text if ch in _SINDHI_LETTERS) >= 3:
            lang, method, conf = "Sindhi", "script + Sindhi-only letters", "medium"
        elif sum(1 for ch in text if ch in _KASHMIRI_LETTERS) >= 2:
            lang, method, conf = "Kashmiri", "script + Kashmiri-only letters", "medium"
    return {"language": lang, "code": LANG_CODES[lang], "script": SCRIPT_NAMES[block], "script_block": block,
            "method": method, "confidence": conf}

def detect_bill_language(text):
    """Dominant script/language -> (language name, IndicTrans2 code). Same signature as v2."""
    d = detect_language_details(text)
    return d["language"], d["code"]

def line_language_code(line, bill):
    """IndicTrans2 code for one line: the bill's language if the line uses the bill's script, else the script default."""
    counts = _script_counts(_norm(line))
    if not counts:
        return bill["code"]
    block = max(counts, key=counts.get)
    return bill["code"] if block == bill.get("script_block") else LANG_CODES.get(block, "hin_Deva")

print("Helpers ready | transliteration backend:", "indic-transliteration" if _sanscript else "NOT AVAILABLE",
      "| glossary entries:", len(_GLOSS), "| scripts:", len(_GLOSS_RES))

## Cell 5 — Fast OCR (parallel, language-narrowed, cached)

In [ ]:
import os, io, json, hashlib, time
from pathlib import Path
os.environ.setdefault("OMP_THREAD_LIMIT", "1")      # one thread per Tesseract process; we parallelise bills instead
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageEnhance, ImageOps
import numpy as np
import pytesseract

_t0 = time.time()
OSD_SCRIPT_TO_LANG = {"Devanagari": "hin", "Telugu": "tel", "Kannada": "kan", "Malayalam": "mal",
                      "Bengali": "ben", "Gujarati": "guj", "Gurmukhi": "pan", "Oriya": "ori",
                      "Tamil": "tam", "Arabic": "urd"}
available = set(pytesseract.get_languages(config=""))
default_langs = [c for c in OCR_LANGUAGES if c in available]
missing = [c for c in OCR_LANGUAGES if c not in available]
if missing:
    print("WARNING: OCR language pack(s) not installed, skipped:", missing)
print("Default OCR languages: eng +", default_langs, "| script detection:", AUTO_NARROW_LANGUAGE and "osd" in available)

Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)
_cache_file = Path(CACHE_DIR) / "ocr_cache.json"
try:
    _cache = json.loads(_cache_file.read_text(encoding="utf-8")) if (USE_CACHE and _cache_file.exists()) else {}
except Exception:
    _cache = {}

def prepare_image(img):
    g = ImageOps.grayscale(img)
    scale = 2.0 if g.width < 700 else 1.5 if g.width < 1100 else 1.0     # only enlarge small images
    if scale != 1.0:
        g = g.resize((int(g.width * scale), int(g.height * scale)), Image.Resampling.BILINEAR)
    if max(g.size) > 3200:
        g.thumbnail((3200, 3200))
    g = ImageOps.autocontrast(g, cutoff=1)
    return ImageEnhance.Contrast(g).enhance(1.3)

# Tesseract pack for each detected language (Sindhi/Kashmiri are read with the Urdu pack)
LANG_TO_TESS = {"Hindi": "hin", "Marathi": "mar", "Nepali": "nep", "Bengali": "ben", "Assamese": "asm",
                "Gujarati": "guj", "Punjabi": "pan", "Odia": "ori", "Tamil": "tam", "Telugu": "tel",
                "Kannada": "kan", "Malayalam": "mal", "Urdu": "urd", "Sindhi": "urd", "Kashmiri": "urd"}

print("Indian OCR packs installed:", sorted(set(LANG_TO_TESS.values()) & available))

def _osd_script(img, size):
    small = ImageOps.grayscale(img)
    small.thumbnail((size, size))
    osd = pytesseract.image_to_osd(small, config="--psm 0 -c min_characters_to_try=10",
                                   output_type=pytesseract.Output.DICT)
    return osd.get("script")

def _probe_langs(img):
    """Script detection failed twice: one quick low-resolution pass with every installed pack, only to read the script."""
    pool = sorted(set(LANG_TO_TESS.values()) & available)
    if len(pool) <= len(default_langs):
        return None
    try:
        small = ImageOps.grayscale(img)
        small.thumbnail((1400, 1400))
        text = pytesseract.image_to_string(small, lang="+".join(["eng"] + pool), config="--oem 1 --psm 6")
    except Exception:
        return None
    code = LANG_TO_TESS.get(detect_language_details(text)["language"])
    return [code] if code in available else None

def choose_langs(img):
    """-> (languages, how). Same behaviour as v2 when the script is detected; extra fallbacks only when detection fails."""
    if AUTO_NARROW_LANGUAGE and "osd" in available:
        for size in (1000, 2000):                  # a larger copy is tried only if the first attempt fails
            try:
                script = _osd_script(img, size)
            except Exception:
                continue
            code = OSD_SCRIPT_TO_LANG.get(script)
            if code and code in available:
                return [code], f"osd:{script}"
            if script:                             # Latin etc. = English/bilingual bill: keep the fallback list
                return default_langs, f"osd:{script}"
        probe = _probe_langs(img)
        if probe:
            return probe, "probe"
    return default_langs, "default"

def split_composite(img, parts=3):
    """Cut a wide image at the blank vertical gaps nearest to equal thirds (falls back to equal thirds)."""
    w, h = img.size
    col_std = np.asarray(ImageOps.grayscale(img), dtype=np.float32).std(axis=0)
    empty = col_std < 4.0
    cuts = [0]
    for k in range(1, parts):
        target, win = k * w // parts, int(0.08 * w)
        lo, hi = max(1, target - win), min(w - 1, target + win)
        runs, start = [], None
        for x in range(lo, hi):
            if empty[x] and start is None:
                start = x
            elif not empty[x] and start is not None:
                runs.append((start, x - 1)); start = None
        if start is not None:
            runs.append((start, hi - 1))
        cuts.append(min(((a + b) // 2 for a, b in runs), key=lambda c: abs(c - target)) if runs else target)
    cuts.append(w)
    return [img.crop((cuts[i], 0, cuts[i + 1], h)) for i in range(parts)], cuts

bill_images = []
for name in sorted(uploaded_images):
    try:
        image = Image.open(io.BytesIO(uploaded_images[name]))
        image = ImageOps.exif_transpose(image).convert("RGB")
    except Exception as error:
        print("Could not open", name, ":", error)
        continue
    if SPLIT_COMPOSITE and COMPOSITE_KEYWORD.lower() in name.lower():
        crops, cuts = split_composite(image)
        print(f"Composite {name!r} split at x = {cuts[1:-1]} (image width {image.width})")
        for i, (crop, label) in enumerate(zip(crops, ["UP_Bill", "MP_Bill", "Delhi_Bill"]), 1):
            bill_images.append({"file_name": name, "identifier": f"{Path(name).stem}_{label}_{i}", "image": crop,
                                "hash": hashlib.sha1(uploaded_images[name] + bytes([i])).hexdigest()})
    else:
        bill_images.append({"file_name": name, "identifier": Path(name).stem, "image": image,
                            "hash": hashlib.sha1(uploaded_images[name]).hexdigest()})

def ocr_one(item):
    started = time.time()
    langs, how = choose_langs(item["image"])
    lang_str = "+".join(["eng"] + langs)
    key = f"{item['hash']}|{lang_str}"
    if USE_CACHE and key in _cache:
        return item, _cache[key], _cache.get(key + "#used", lang_str), 0.0, True, how
    text = pytesseract.image_to_string(prepare_image(item["image"]), lang=lang_str, config="--oem 1 --psm 6").strip()
    used = lang_str
    # Languages that share a script (Hindi/Marathi/Nepali, Bengali/Assamese): if the text says it is another
    # language of that script, read the bill once more with that language's pack. Other bills are not re-read.
    want = LANG_TO_TESS.get(detect_language_details(text)["language"]) if text else None
    if want and want in available and want not in langs:
        again = pytesseract.image_to_string(prepare_image(item["image"]), lang="eng+" + want, config="--oem 1 --psm 6").strip()
        if again:
            text, used = again, "eng+" + want
    if USE_CACHE:
        _cache[key] = text
        _cache[key + "#used"] = used
    return item, text, used, time.time() - started, False, how

workers = MAX_WORKERS or max(1, os.cpu_count() or 1)
records, failed = [], []
with ThreadPoolExecutor(max_workers=min(workers, max(1, len(bill_images)))) as pool:
    for fut in [pool.submit(ocr_one, it) for it in bill_images]:
        try:
            item, text, lang_str, secs, cached, how = fut.result()
        except Exception as error:
            failed.append(str(error)); print("OCR failed:", error); continue
        _cache[f"{item['hash']}|{lang_str}"] = text
        records.append({"record_id": f"BILL_{len(records) + 1:04d}", "file_name": item["file_name"],
                        "identifier": item["identifier"], "original_ocr_text": text,
                        "ocr_languages_used": lang_str, "ocr_language_detection": how, "source_language": None,
                        "english_translation": None, "translation_status": "pending"})
        print(f"OCR {item['identifier']:<45} {lang_str:<12} {'cached' if cached else f'{secs:5.1f}s'} | {len(text)} chars | {how}")

if USE_CACHE:
    try: _cache_file.write_text(json.dumps(_cache, ensure_ascii=False), encoding="utf-8")
    except Exception as error: print("Cache not saved:", error)
if not records:
    raise RuntimeError("No bills were processed successfully.")
print(f"OCR done: {len(records)} bills in {time.time() - _t0:.1f}s")

## Cell 6 — Translate to English (glossary → optional IndicTrans2 in batches)

* Words from the glossary are translated instantly.
* A line that already has English next to the Indian text (typical for Karnataka bills) simply drops the Indian duplicate.
* The language of each bill is detected automatically; the JSON always contains the original text, the English translation and (for every Indian-script line) the transliteration.
* In `"fast"` mode any other Indian-script word is transliterated into Roman letters. In `"hybrid"` mode only those leftover lines go to IndicTrans2, in sorted batches.

In [ ]:
_t0 = time.time()
_LATIN_WORD = re.compile(r"[A-Za-z]{3,}")
# scripts the rule-based transliterator cannot handle (Urdu/Sindhi/Kashmiri, Santali, Manipuri)
_UNCOVERED_RE = re.compile(r"[\u0600-\u06ff\u1c50-\u1c7f\uabc0-\uabff]")

def _drop_indic_if_bilingual(line):
    """If the line already has real English words, remove the Indian-script duplicate."""
    if INDIC_RE.search(line) and len(_LATIN_WORD.findall(line)) >= 2:
        return re.sub(r"\s{2,}", " ", INDIC_RE.sub("", re.sub(f"[{INDIC_CHARS}]+", " ", line))).strip(" /|-:")
    return line

model = tokenizer = indic_processor = None
device = "cpu"

def _ensure_model_packages():
    """Install the model packages the first time they are needed (no-op in hybrid/model mode: Cell 2 did it)."""
    needed = [p for p, m in [("transformers", "transformers"), ("sentencepiece", "sentencepiece"),
                             ("sacremoses", "sacremoses")] if not _have(m)]
    if needed:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install"] + needed, check=True)
    if not _have("IndicTransToolkit"):
        subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                        "git+https://github.com/VarunGumma/IndicTransToolkit.git"], check=True)

def load_translation_model():
    global model, tokenizer, indic_processor, device
    if model is not None:
        return
    _ensure_model_packages()
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    from IndicTransToolkit import IndicProcessor
    name = "ai4bharat/indictrans2-indic-en-dist-200M"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    started = time.time()
    tokenizer = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    m = AutoModelForSeq2SeqLM.from_pretrained(name, trust_remote_code=True, torch_dtype=dtype).to(device).eval()
    indic_processor = IndicProcessor(inference=True)
    if device == "cpu" and CPU_QUANTIZE:
        try:
            q = torch.quantization.quantize_dynamic(m, {torch.nn.Linear}, dtype=torch.qint8)
            _translate_batch(["नमस्ते"], "hin_Deva", q)            # smoke test; falls back if it fails
            m = q
            print("CPU int8 quantisation enabled")
        except Exception as error:
            print("Quantisation skipped:", error)
    model = m
    print(f"IndicTrans2 loaded on {device} in {time.time() - started:.1f}s")
    if device == "cpu":
        print("TIP: Runtime > Change runtime type > T4 GPU makes model translation much faster.")

def _translate_batch(batch, src_code, mdl=None):
    import torch
    mdl = mdl or model
    pre = indic_processor.preprocess_batch(batch, src_lang=src_code, tgt_lang="eng_Latn")
    inputs = tokenizer(pre, return_tensors="pt", padding="longest", truncation=True, max_length=256).to(device)
    with torch.inference_mode():
        out = mdl.generate(**inputs, use_cache=True, min_length=0, max_length=256, num_beams=1, num_return_sequences=1)
    dec = tokenizer.batch_decode(out, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    return indic_processor.postprocess_batch(dec, lang="eng_Latn")

def model_translate(lines, src_code):
    batch_size = 24 if device == "cuda" else 8
    order = sorted(range(len(lines)), key=lambda i: len(lines[i]))      # similar lengths = less padding
    result = [None] * len(lines)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        try:
            for i, t in zip(idx, _translate_batch([lines[i] for i in idx], src_code)):
                result[i] = t
        except Exception as error:
            print("Batch failed, using transliteration for those lines:", error)
    return result

# ---- pass 1: language detection, glossary / bilingual / transliteration; collect lines that need the model ----
plan, methods, pending = {}, {}, {}   # plan[(record_idx, line_idx)] = text ; pending[lang_code] = [(key, text)]
bill_lang = {}
for ri, rec in enumerate(records):
    det = detect_language_details(rec["original_ocr_text"])
    bill_lang[ri] = det
    rec["source_language"] = det["language"]
    rec["source_language_code"] = det["code"]
    rec["source_script"] = det["script"]
    rec["language_detection_method"] = det["method"]
    rec["language_detection_confidence"] = det["confidence"]
    rec["output_language"] = "English"
    rec["translation_method"] = TRANSLATION_MODE
    for li, raw in enumerate((rec["original_ocr_text"] or "").splitlines()):
        line = raw.strip()
        if not line:
            plan[(ri, li)] = ""; continue
        if not (INDIC_RE.search(line) or ARABIC_RE.search(line)):
            plan[(ri, li)] = line; continue
        line = ascii_digits(line)        # native digits -> 0-9 so numbers survive translation (original text is kept as is)
        if TRANSLATION_MODE == "model":
            text = _drop_indic_if_bilingual(line)
        else:
            text = _drop_indic_if_bilingual(apply_glossary(line))
        if not (INDIC_RE.search(text) or ARABIC_RE.search(text)):
            plan[(ri, li)] = text
            methods[(ri, li)] = "bilingual line" if TRANSLATION_MODE == "model" else "glossary / bilingual line"
            continue
        use_model = TRANSLATION_MODE in ("hybrid", "model") or (AUTO_MODEL_FOR_UNCOVERED and _UNCOVERED_RE.search(text))
        if not use_model:
            plan[(ri, li)] = transliterate_text(text)
            methods[(ri, li)] = "glossary + transliteration"
        else:
            pending.setdefault(line_language_code(text, det), []).append(((ri, li), text))

# ---- pass 2: one batched model run for everything that is left --------------------------------
if pending:
    total = sum(len(v) for v in pending.values())
    print(f"{total} line(s) need IndicTrans2")
    try:
        load_translation_model()
        for code_, items in pending.items():
            unique = list(dict.fromkeys(t for _k, t in items))              # translate repeated lines once
            done = dict(zip(unique, model_translate(unique, code_)))
            for key, text in items:
                out = done.get(text)
                plan[key] = out or transliterate_text(text)
                methods[key] = "IndicTrans2" if out else "transliteration (model returned nothing)"
    except Exception as error:
        print("IndicTrans2 is not available, using transliteration for the remaining lines:", error)
    for code_, items in pending.items():                                    # anything the model did not cover
        for key, text in items:
            if key not in plan:
                plan[key] = transliterate_text(text)
                methods[key] = "transliteration (model unavailable)"

# ---- helpers for short fields (names / addresses), used again in Cell 7 ----------------------
def translate_snippet(text, code="hin_Deva"):
    """English for a short field. Returns (english, method). The model is used only if it is already loaded."""
    text = ascii_digits(text) if text else text
    if not text or not (INDIC_RE.search(text) or ARABIC_RE.search(text)):
        return text, "no translation needed"
    t = _drop_indic_if_bilingual(text if TRANSLATION_MODE == "model" else apply_glossary(text))
    if not (INDIC_RE.search(t) or ARABIC_RE.search(t)):
        return t, "glossary / bilingual line"
    if model is not None:
        try:
            out = _translate_batch([t], code)[0]
            if out:
                return out, "IndicTrans2"
        except Exception:
            pass
    return transliterate_text(t), "transliteration only"

def romanize_field(text, code="hin_Deva"):
    """Name/address -> Roman letters. Rule-based first; scripts it cannot do use the model if it is loaded."""
    out = transliterate_text(text)
    if out and (INDIC_RE.search(out) or ARABIC_RE.search(out)) and model is not None:
        try:
            res = _translate_batch([text], code)[0]
            if res:
                return res, "IndicTrans2 (model)"
        except Exception:
            pass
    return out, "indic-transliteration (rule-based, Roman letters)"

for ri, rec in enumerate(records):
    lines = (rec["original_ocr_text"] or "").splitlines()
    rec["english_translation"] = "\n".join(plan.get((ri, li), "") for li in range(len(lines))).strip()
    rec["translation_status"] = "success" if rec["english_translation"] else "empty_ocr_text"
    # Every line that contains Indian-script text, side by side: original | English | transliteration
    pairs = []
    for li, raw in enumerate(lines):
        raw = raw.strip()
        if raw and (INDIC_RE.search(raw) or ARABIC_RE.search(raw)):
            roman = transliterate_text(raw)
            pairs.append({"line_no": li + 1, "original": raw, "english_translation": plan.get((ri, li), ""),
                          "transliteration": None if (INDIC_RE.search(roman or "") or ARABIC_RE.search(roman or "")) else roman,
                          "method": methods.get((ri, li), "")})
    rec["translated_lines"] = pairs
    print(f"{ri + 1}/{len(records)} {rec['identifier']:<45} {rec['source_language']:<9} -> {rec['translation_status']}")
print(f"Translation done in {time.time() - _t0:.1f}s (mode: {TRANSLATION_MODE})")


## Cell 7 — Extract bill fields (fixed patterns) + transliterated names/addresses

In [ ]:
_t0 = time.time()
SEP = r"\s*[:#：\-–—.%=§$¢~+|]*\s*"
AMT = r"(?:₹|Rs\.?|INR)?\s*(-?[\d,]+(?:\.\d{1,2})?)"
DATE = r"(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})"
UNITS = r"(\d+(?:\.\d+)?)"

FIELD_PATTERNS = {
    "consumer_number": [r"\bUSC\s*(?:NO\.?)?\s*[:.]?\s*([0-9][0-9\-]{4,})", r"consumer\s*(?:no\.?|number)" + SEP + r"((?=[A-Z0-9\-]*\d)[A-Z0-9][A-Z0-9\-]{4,})"],
    "service_connection_number": [r"\bSC\s*(?:NO\.?)?\s*[:.]?\s*([0-9][0-9\-]{4,})", r"service\s*connection\s*(?:no\.?|number)" + SEP + r"((?=[A-Z0-9\-]*\d)[A-Z0-9][A-Z0-9\-]{3,})"],
    "customer_id": [r"customer\s*id" + SEP + r"(\d+)"],
    "meter_number": [r"\bMtr\s*no\.?\s*[:.]?\s*([A-Z0-9]{5,})", r"\bMeter\s*No\.?\s*[:.]?\s*([A-Z0-9]{5,})", r"meter\s*number" + SEP + r"((?=[A-Z0-9\-]*\d)[A-Z0-9][A-Z0-9\-]{4,})"],
    "bill_number": [r"\bBill\s*No\.?\s*[:.]?\s*(\d+)"],
    "bill_date": [r"bill\s*date" + SEP + DATE, r"\b(?:Dt|DL)\s*[:.]\s*" + DATE, r"bill\s*date[^\n]*\n[^\n]*?" + DATE],
    "due_date": [r"due\s*date" + SEP + DATE, r"(?<!the )due\s*date[^\n]*\n[^\n]*?" + DATE],
    "last_payment_date": [r"last\s*paid\s*(?:dt|date)" + SEP + DATE, r"last\s*payment\s*date" + SEP + DATE],
    "last_payment_amount": [r"last\s*paid\s*(?:amt|amount)" + SEP + AMT, r"last\s*payment\s*amount" + SEP + AMT],
    "units_consumed": [r"consum\w{3,9}\s*\(\s*units\s*\)" + SEP + UNITS, r"\bKWH\s*Un\w{2,4}" + SEP + UNITS,
                       r"(?<!Month\s)(?<!Monthly\s)(?<!Entitlement\s)\bunits\s*(?:consumed)?" + SEP + UNITS],
    "current_bill_amount": [r"(?<!net\s)bill\s*a[a-z]{4,7}" + SEP + AMT, r"total\s*current\s*due" + SEP + AMT, r"total\s*amount\s*due\s*\(current\s*month\)" + SEP + AMT,
                            r"cur\.?\s*demand\s*payable" + SEP + AMT],
    "previous_balance": [r"previous\s*(?:balance|arrears|dues?)" + SEP + AMT, r"\barrears\s*[:\-]?\s*(-?[\d,]+(?:\.\d{1,2})?)"],
    "late_fee": [r"late\s*(?:payment\s*)?(?:fee|surcharges?)" + SEP + AMT],
    "subsidy": [r"(?:GJS|Gruha\s*Jyoth\w*)\s*subs\w*" + SEP + AMT, r"\bsubsidy" + SEP + AMT],
    "net_bill_amount": [r"net\s*bill\s*a[a-z]{4,7}" + SEP + AMT, r"net\s*bill\b" + SEP + AMT, r"bill\s*aft\w*\s*sub\w*" + SEP + AMT,
                         r"total\s*current\s*due" + SEP + AMT, r"\bne[a-z]\s*payable" + SEP + AMT],
    "total_amount_due": [r"total\s*amount\s*due(?!\s*\(current)" + SEP + AMT, r"total\s*due" + SEP + AMT, r"\bne[a-z]\s*payable" + SEP + AMT,
                         r"amount\s*payable" + SEP + AMT, r"\bnet\s*bill\b(?!\s*a[a-z]{4,7})" + SEP + AMT,
                         r"bill\s*aft\w*\s*sub\w*" + SEP + AMT],
}
NUMERIC_FIELDS = {"units_consumed", "current_bill_amount", "previous_balance", "late_fee", "subsidy",
                  "net_bill_amount", "total_amount_due", "last_payment_amount"}

PROVIDERS = [
    (r"TGNPDCL|TSNPDCL|northern\s*power", "Northern Power Distribution Company of Telangana Limited (TGNPDCL)"),
    (r"TGSPDCL|TSSPDCL|southern\s*power", "Southern Power Distribution Company of Telangana Limited (TGSPDCL)"),
    (r"UPPCL|UTTAR\s*PRADESH\s*POWER|उत्तर\s*प्रदेश\s*पावर", "Uttar Pradesh Power Corporation Limited (UPPCL)"),
    (r"MPWZ|मध्य\s*प्रदेश\s*पश|Madhya\s*Pradesh\s*(?:West|Paschim)", "Madhya Pradesh Paschim Kshetra Vidyut Vitaran Company Limited"),
    (r"BSES|बी\s*एस\s*ई\s*एस", "BSES Rajdhani Power Limited"),
    (r"BESCOM|HESCOM|MESCOM|GESCOM|CESC", "Karnataka ESCOM (verify on bill)"),
    (r"GJS\s*Reg\s*Date", "Karnataka ESCOM (verify on bill)"),
    (r"\bUSC\b|\bERO\b", "Telangana DISCOM - TGSPDCL or TGNPDCL (verify on bill)"),
]
PROVIDERS += [
    (r"MSEDCL|Mahavitaran|महावितरण", "Maharashtra State Electricity Distribution Company Limited (MSEDCL)"),
    (r"WBSEDCL", "West Bengal State Electricity Distribution Company Limited (WBSEDCL)"),
    (r"TANGEDCO|TNPDCL|\bTNEB\b", "Tamil Nadu Power Distribution Corporation Limited (TNPDCL, formerly TANGEDCO)"),
    (r"\bKSEB", "Kerala State Electricity Board Limited (KSEB)"),
    (r"PSPCL", "Punjab State Power Corporation Limited (PSPCL)"),
    (r"UGVCL|DGVCL|MGVCL|PGVCL|\bGUVNL\b", "Gujarat DISCOM - UGVCL / DGVCL / MGVCL / PGVCL (verify on bill)"),
    (r"TPDDL|Tata\s*Power\s*Delhi", "Tata Power Delhi Distribution Limited (TPDDL)"),
    (r"TPCODL|TPNODL|TPSODL|TPWODL|\bNESCO\b|\bWESCO\b|\bSOUTHCO\b|\bCESU\b", "Odisha DISCOM (verify on bill)"),
    (r"APSPDCL|APEPDCL|APCPDCL", "Andhra Pradesh DISCOM - APSPDCL / APEPDCL / APCPDCL (verify on bill)"),
]
LIMITED_SUPPORT = {"Santali", "Manipuri", "Sindhi", "Kashmiri"}     # no dedicated Tesseract model / rule-based transliteration
MONTHS = r"(?:January|February|March|April|May|June|July|August|September|October|November|December|जनवरी|फरवरी|मार्च|अप्रैल|मई|जून|जुलाई|अगस्त|सितम्बर|सितंबर|अक्टूबर|नवंबर|दिसंबर)"
MONTHS = MONTHS[:-1] + "|" + "|".join(re.escape(w) for w in sorted(MONTH_WORDS_N, key=len, reverse=True)) + ")"
LABEL_STOP = re.compile(r"\s*(?:बिल\s*(?:दिनांक|माह)|देय\s*तिथि|Bill\s*(?:date|month)|वितरण|कनेक्शन|कनेकशन|वृत्त|\|)", re.I)
NAME_LABEL = r"(?:\b(?:customer|consumer)\s*name\b|\bname\b|नाम|పేరు|ಹೆಸರು|नाव|নাম|નામ|பெயர்|പേര്|ਨਾਮ|ନାମ|نام)"
ADDR_LABEL = r"(?:\baddress\b|\baddr\b|पता|చిరునామా|ವಿಳಾಸ|पत्ता|ঠিকানা|સરનામું|முகவரி|വിലാസം|ਪਤਾ|ଠିକଣା|پتہ)"

def _gloss_lookup(word):
    n = _norm(word)
    return _GLOSS.get(n) or MONTH_WORDS_N.get(n) or word

def amount(value):
    if value is None:
        return None
    value = str(value).strip()
    if re.fullmatch(r"-?\d+,\d{2}", value):          # OCR often reads the decimal point as a comma: 715,00
        value = value.replace(",", ".")
    value = re.sub(r"[^0-9.\-]", "", value.replace(",", ""))
    try:
        return float(value) if value not in ("", "-", ".") else None
    except ValueError:
        return None

def _first(patterns, text):
    for p in patterns:
        m = re.search(p, text, re.I)
        if m:
            return m.group(1).strip()
    return None

def _clean_value(s):
    s = re.sub(r"\s*[=_|]+\s*", " ", s).strip(" -")
    words = s.split()
    while words and re.search(r"[\[\]{}<>“”\"=|~_*^§$¢]", words[-1]):   # drop trailing OCR junk such as "[ee"
        words.pop()
    return re.sub(r"\s{2,}", " ", " ".join(words))

def extract_name(original):
    m = re.search(NAME_LABEL + r"\s*[:：]\s*([^\n]+)", original, re.I)
    if not m:
        return None
    name = LABEL_STOP.split(m.group(1))[0]
    name = re.sub(rf"\s+{MONTHS}\s+20\d{{2}}.*$", "", name, flags=re.I)
    return _clean_value(name)[:100] or None

def extract_address(original):
    lines = original.splitlines()
    for i, line in enumerate(lines):
        m = re.search(ADDR_LABEL + r"\s*[:：]\s*(.+)", line, re.I)
        if not m:
            continue
        parts = [LABEL_STOP.split(m.group(1))[0]]
        if i + 1 < len(lines):                       # addresses often continue on the next line
            nxt = lines[i + 1].strip().lstrip("|").strip()
            if nxt and not re.search(r"[:：]", nxt) and len(nxt) < 90:
                parts.append(LABEL_STOP.split(nxt)[0])
        return _clean_value(", ".join(p.strip(" ,") for p in parts if p.strip(" ,")))[:250] or None
    return None

def extract_fields(record):
    original = ascii_digits(_norm(record.get("original_ocr_text") or ""))
    english = ascii_digits(record.get("english_translation") or "")
    combined = original + "\n" + english

    f = {k: None for k in ["electricity_provider", "consumer_number", "customer_id", "service_connection_number",
                           "customer_name", "address", "bill_number", "bill_date", "bill_month", "due_date",
                           "meter_number", "units_consumed", "current_bill_amount", "previous_balance", "late_fee",
                           "subsidy", "net_bill_amount", "total_amount_due", "last_payment_date", "last_payment_amount"]}
    for pattern, provider in PROVIDERS:
        if re.search(pattern, combined, re.I):
            f["electricity_provider"] = provider; break
    for field, patterns in FIELD_PATTERNS.items():
        value = _first(patterns, english) or _first(patterns, combined)
        if value is not None:
            f[field] = amount(value) if field in NUMERIC_FIELDS else value
    m = re.search(rf"({MONTHS})\s*(20\d{{2}})", original, re.I) or re.search(rf"({MONTHS})\s*(20\d{{2}})", english, re.I)
    if m:
        f["bill_month"] = f"{_norm(m.group(1)) if m.group(1).isascii() else _gloss_lookup(m.group(1))} {m.group(2)}"

    f["customer_name"] = extract_name(original)
    f["address"] = extract_address(original)
    # Names/addresses: original is kept; Roman-letter transliteration; address also gets an English translation.
    code = record.get("source_language_code") or "hin_Deva"
    f["customer_name_english"], _m1 = romanize_field(f["customer_name"], code)
    f["address_english"], _m2 = romanize_field(f["address"], code)
    f["name_address_conversion"] = ("IndicTrans2 (model) for scripts the rule-based transliterator cannot do"
                                    if "IndicTrans2" in (_m1 + _m2) else "indic-transliteration (rule-based, Roman letters)")
    _addr_en, _addr_how = translate_snippet(f["address"], code)
    f["text_fields"] = {
        "customer_name": {"original": f["customer_name"], "transliteration": f["customer_name_english"],
                          "english_translation": None,
                          "note": "Names are transliterated (Roman letters), not translated."},
        "address": {"original": f["address"], "transliteration": f["address_english"],
                    "english_translation": _addr_en, "translation_method": _addr_how},
    }

    flags = []
    if f["total_amount_due"] is None and f["current_bill_amount"] is not None:
        f["total_amount_due"] = round(f["current_bill_amount"] + (f["previous_balance"] or 0) + (f["late_fee"] or 0), 2)
        flags.append("Total amount due was calculated (current + previous + late fee), not read directly. Verify.")
    if not f["electricity_provider"]:
        flags.append("Provider not confidently identified from OCR.")
    if not (f["consumer_number"] or f["customer_id"] or f["service_connection_number"]):
        flags.append("No customer/account identifier confidently extracted.")
    if f["total_amount_due"] is None:
        flags.append("Total amount due not confidently extracted.")
    if record.get("source_language") in LIMITED_SUPPORT:
        flags.append(f"{record['source_language']}: no dedicated OCR model or rule-based transliteration in this notebook; text may be unreliable. Verify against the bill image.")
    for k in ("customer_name_english", "address_english"):
        if f[k] and (INDIC_RE.search(f[k]) or ARABIC_RE.search(f[k])):
            flags.append(f"{k} still contains non-Roman letters (transliteration unavailable for that script).")
    f["review_flags"] = flags
    f["manual_review_required"] = bool(flags)
    return f

for record in records:
    record.update(extract_fields(record))
print(f"Field extraction done in {time.time() - _t0:.2f}s. Verify every amount, date and name against the bill image.")

## Cell 8 — Save JSON + CSV and show a short summary

In [ ]:
import csv

out_dir = Path(OUTPUT_DIR); out_dir.mkdir(parents=True, exist_ok=True)
json_file = out_dir / "electricity_bills_with_english.json"
csv_file = out_dir / "electricity_bills_summary.csv"

with open(json_file, "w", encoding="utf-8") as fh:
    json.dump({"output_language": "English", "record_count": len(records),
               "languages_detected": {l: sum(1 for r in records if r["source_language"] == l) for l in sorted({r["source_language"] for r in records})},
               "records": records}, fh, ensure_ascii=False, indent=2)
with open(json_file, encoding="utf-8") as fh:
    assert len(json.load(fh)["records"]) == len(records)

SUMMARY_COLS = ["record_id", "identifier", "source_language", "electricity_provider", "customer_name_english",
                "address_english", "consumer_number", "customer_id", "service_connection_number", "meter_number",
                "bill_number", "bill_month", "bill_date", "due_date", "units_consumed", "current_bill_amount",
                "subsidy", "net_bill_amount", "total_amount_due", "last_payment_date", "last_payment_amount",
                "manual_review_required", "source_script", "language_detection_confidence"]
with open(csv_file, "w", encoding="utf-8-sig", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=SUMMARY_COLS, extrasaction="ignore")
    w.writeheader(); w.writerows(records)

print("Valid JSON: True | bills:", len(records))
for r in records:
    print("\n" + "=" * 78)
    print(f"{r['record_id']}  {r['identifier']}  [{r['source_language']}, OCR {r.get('ocr_languages_used')}]")
    for k in SUMMARY_COLS[3:]:
        if r.get(k) not in (None, ""):
            print(f"  {k:<26} {r[k]}")
    for flag in r["review_flags"]:
        print("  ! REVIEW:", flag)

try:
    from google.colab import files
    files.download(str(json_file)); files.download(str(csv_file))
except ImportError:
    print("\nSaved to", out_dir)